# Whole Human Brain Region Exploration

Explore the Allen Whole Human Brain metadata used by MapMyCells: available regions, cell counts by region, and how region labels relate to anatomical/gyrus-style names and Brodmann-style area labels.

This notebook downloads only metadata CSVs, not the raw expression H5ADs. The large file is `cell_metadata.csv` (~830 MB), cached under the same MapMyCells cache root used by the pipeline.

In [ ]:
from __future__ import annotations

import json
import re
import urllib.parse
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

CACHE_DIR = Path("/media/mathieubo/SSD2/MerXen/mapmycells")
ABC_CACHE_DIR = CACHE_DIR / "abc_whb"
MANIFEST_URL = "https://allen-brain-cell-atlas.s3.us-west-2.amazonaws.com/releases/20250531/manifest.json"

ABC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

## Download or Reuse Metadata

The helper below checks the manifest-reported file size and reuses existing cached files when possible.

In [ ]:
def load_manifest(url: str = MANIFEST_URL) -> dict:
    with urllib.request.urlopen(url) as response:
        return json.loads(response.read().decode("utf-8"))


def manifest_file_info(manifest: dict, *keys: str) -> dict:
    node = manifest["file_listing"]
    for key in keys:
        node = node[key]
    return dict(node)


def ensure_manifest_file(info: dict, cache_dir: Path = ABC_CACHE_DIR) -> Path:
    output_path = cache_dir / info["relative_path"]
    expected_size = int(info.get("size", 0) or 0)
    if output_path.exists() and (not expected_size or output_path.stat().st_size == expected_size):
        return output_path

    output_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = output_path.with_name(output_path.name + ".tmp")
    print(f"Downloading {info['url']} -> {output_path}")
    with urllib.request.urlopen(info["url"]) as response, tmp_path.open("wb") as out:
        while True:
            chunk = response.read(16 * 1024 * 1024)
            if not chunk:
                break
            out.write(chunk)

    actual_size = tmp_path.stat().st_size
    if expected_size and actual_size != expected_size:
        tmp_path.unlink(missing_ok=True)
        raise RuntimeError(f"Downloaded size {actual_size:,} does not match expected {expected_size:,}")
    tmp_path.replace(output_path)
    return output_path


manifest = load_manifest()
cell_metadata_path = ensure_manifest_file(
    manifest_file_info(manifest, "WHB-10Xv3", "metadata", "cell_metadata", "files", "csv")
)
roi_map_path = ensure_manifest_file(
    manifest_file_info(manifest, "WHB-10Xv3", "metadata", "region_of_interest_structure_map", "files", "csv")
)
division_map_path = ensure_manifest_file(
    manifest_file_info(manifest, "WHB-10Xv3", "metadata", "anatomical_division_structure_map", "files", "csv")
)

cell_metadata_path, roi_map_path, division_map_path

## Region and Anatomical Division Tables

In [ ]:
roi_map = pd.read_csv(roi_map_path)
division_map = pd.read_csv(division_map_path)

display(roi_map.head())
display(division_map.head())
print(f"ROI map rows: {len(roi_map):,}")
print(f"Unique ROI labels: {roi_map['region_of_interest_label'].nunique():,}")

## Count Cells by Region

This uses chunked reads so you do not need to hold the whole metadata table in memory.

In [ ]:
def count_metadata_groups(path: Path, group_cols: list[str], chunksize: int = 1_000_000) -> pd.DataFrame:
    parts = []
    for chunk in pd.read_csv(path, usecols=group_cols, chunksize=chunksize):
        parts.append(chunk.groupby(group_cols, dropna=False).size())
    counts = pd.concat(parts).groupby(level=list(range(len(group_cols)))).sum()
    return counts.rename("n_cells").reset_index().sort_values("n_cells", ascending=False)


region_counts = count_metadata_groups(cell_metadata_path, ["region_of_interest_label"])
division_counts = count_metadata_groups(cell_metadata_path, ["anatomical_division_label"])
region_by_division = count_metadata_groups(cell_metadata_path, ["anatomical_division_label", "region_of_interest_label"])
region_by_feature_matrix = count_metadata_groups(cell_metadata_path, ["region_of_interest_label", "feature_matrix_label"])

display(region_counts.head(30))
display(division_counts)

In [ ]:
ax = region_counts.head(40).sort_values("n_cells").plot.barh(
    x="region_of_interest_label",
    y="n_cells",
    figsize=(9, 11),
    legend=False,
)
ax.set_xlabel("Cells")
ax.set_ylabel("WHB region_of_interest_label")
ax.set_title("Top 40 WHB regions by cell count")
plt.tight_layout()

## Build a Region Crosswalk

The WHB metadata uses `region_of_interest_label` as the cell-level region. The ROI structure map tells you which DHBA structures each ROI label points to. Some structures are Brodmann-like (`A46`, `A44`, `A45`); others are gyrus-like (`MTG`, `STG`) or named cortical areas.

In [ ]:
def unique_join(values: pd.Series) -> str:
    cleaned = sorted({str(v) for v in values.dropna() if str(v).strip()})
    return "; ".join(cleaned)


def extract_brodmann_terms(symbols: str, names: str) -> str:
    terms = set(re.findall(r"\bA\d+[A-Za-z]*\b", symbols or ""))
    for match in re.findall(r"area\s+([0-9]+[A-Za-z]?)", names or "", flags=re.IGNORECASE):
        terms.add(f"area {match}")
    return "; ".join(sorted(terms))


def extract_gyrus_terms(names: str) -> str:
    # Pull compact phrases ending in "gyrus"; keep this intentionally simple for inspection.
    terms = re.findall(r"([A-Za-z][A-Za-z\- ]+? gyrus)", names or "", flags=re.IGNORECASE)
    return "; ".join(sorted({term.strip() for term in terms}))


division_by_region = (
    region_by_division.groupby("region_of_interest_label", as_index=False)
    .agg(anatomical_division_label=("anatomical_division_label", unique_join))
)


roi_crosswalk = (
    roi_map.groupby("region_of_interest_label", as_index=False)
    .agg(
        structure_symbols=("structure_symbol", unique_join),
        structure_names=("structure_name", unique_join),
        structure_identifiers=("structure_identifier", unique_join),
    )
    .merge(region_counts, on="region_of_interest_label", how="left")
    .merge(division_by_region, on="region_of_interest_label", how="left")
    .sort_values("n_cells", ascending=False)
)

roi_crosswalk["brodmann_terms"] = roi_crosswalk.apply(
    lambda row: extract_brodmann_terms(row["structure_symbols"], row["structure_names"]),
    axis=1,
)
roi_crosswalk["gyrus_terms"] = roi_crosswalk["structure_names"].map(extract_gyrus_terms)
roi_crosswalk["has_brodmann_like_annotation"] = roi_crosswalk["brodmann_terms"].ne("")
roi_crosswalk["has_gyrus_like_annotation"] = roi_crosswalk["gyrus_terms"].ne("")

display(roi_crosswalk.head(30))

In [ ]:
annotation_summary = (
    roi_crosswalk.groupby(["has_brodmann_like_annotation", "has_gyrus_like_annotation"], dropna=False)
    .agg(n_regions=("region_of_interest_label", "nunique"), n_cells=("n_cells", "sum"))
    .reset_index()
    .sort_values("n_cells", ascending=False)
)
display(annotation_summary)

display(
    roi_crosswalk.loc[
        roi_crosswalk["has_brodmann_like_annotation"] | roi_crosswalk["has_gyrus_like_annotation"],
        [
            "region_of_interest_label",
            "n_cells",
            "anatomical_division_label",
            "structure_symbols",
            "brodmann_terms",
            "gyrus_terms",
            "structure_names",
        ],
    ].sort_values("n_cells", ascending=False)
)

## Search for Regions of Interest

Edit `SEARCH_REGEX` to inspect specific cortical, gyrus, or Brodmann terms. This is useful for deciding which `mapmycells_region_labels` to pass to the pipeline.

In [ ]:
SEARCH_REGEX = r"superior frontal|SFG|frontal|DFC|MFC|OFC|A46|gyrus"

search_columns = [
    "region_of_interest_label",
    "anatomical_division_label",
    "structure_symbols",
    "structure_names",
    "brodmann_terms",
    "gyrus_terms",
]
mask = pd.Series(False, index=roi_crosswalk.index)
for column in search_columns:
    mask |= roi_crosswalk[column].fillna("").str.contains(SEARCH_REGEX, case=False, regex=True)

display(
    roi_crosswalk.loc[
        mask,
        ["region_of_interest_label", "n_cells", *search_columns[1:]],
    ].sort_values("n_cells", ascending=False)
)

## Region Counts Split by Neurons vs Nonneurons

The WHB expression matrices are split into `WHB-10Xv3-Neurons` and `WHB-10Xv3-Nonneurons`. This table helps you see whether a candidate ROI has good coverage in each matrix.

In [ ]:
region_matrix_pivot = (
    region_by_feature_matrix.pivot_table(
        index="region_of_interest_label",
        columns="feature_matrix_label",
        values="n_cells",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
    .merge(region_counts, on="region_of_interest_label", how="left")
    .sort_values("region_of_interest_label", ascending=True)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", 0,
):
    display(region_matrix_pivot)

## SFG Translation Check

The WHB cell metadata stores `region_of_interest_label`, not a direct gyrus label for each cell. This section asks the Allen Human Brain Atlas API about structures such as `SFG`, `A8`, `A9`, `A9/46`, and `A46`, then intersects those structure IDs with the WHB ROI table. This makes it explicit which anatomical concepts are actually available as WHB region labels.

In [ ]:
HUMAN_ATLAS_API = "https://human.brain-map.org/api/v2/data/query.json"


def allen_structure_query(criteria: str) -> pd.DataFrame:
    url = HUMAN_ATLAS_API + "?criteria=" + urllib.parse.quote(criteria, safe=":,[]$'*")
    with urllib.request.urlopen(url) as response:
        payload = json.loads(response.read().decode("utf-8"))
    if not payload.get("success"):
        raise RuntimeError(payload)
    return pd.DataFrame(payload.get("msg", []))


def structures_by_acronym(acronym: str) -> pd.DataFrame:
    return allen_structure_query(f"model::Structure,rma::criteria,[acronym$eq'{acronym}']")


def descendants_of_structure_id(structure_id: int) -> pd.DataFrame:
    criteria = (
        "model::Structure,"
        f"rma::criteria,[structure_id_path$il'*/{structure_id}/*'],"
        "rma::options[order$eq'graph_order'][num_rows$eq'all']"
    )
    return allen_structure_query(criteria)


roi_by_structure_id = roi_map.assign(
    structure_id=roi_map["structure_identifier"].str.replace("DHBA:", "", regex=False).astype(int)
)[["structure_id", "region_of_interest_label", "structure_symbol", "structure_name"]]

candidate_rows = []
for acronym in ["SFG", "MFG", "DFC", "A8", "A9", "A9/46", "A46", "A32"]:
    hits = structures_by_acronym(acronym)
    if hits.empty:
        candidate_rows.append({"query": acronym, "status": "not found in Allen API"})
        continue
    for _, hit in hits.iterrows():
        structure_id = int(hit["id"])
        whb = roi_by_structure_id.loc[roi_by_structure_id["structure_id"] == structure_id]
        candidate_rows.append(
            {
                "query": acronym,
                "allen_structure_id": structure_id,
                "allen_acronym": hit.get("acronym"),
                "allen_name": hit.get("name"),
                "allen_structure_id_path": hit.get("structure_id_path"),
                "available_as_whb_roi": not whb.empty,
                "whb_region_of_interest_label": "; ".join(whb["region_of_interest_label"].astype(str)),
            }
        )

candidate_crosswalk = pd.DataFrame(candidate_rows)
display(candidate_crosswalk)

dfc_hits = structures_by_acronym("DFC")
if not dfc_hits.empty:
    dfc_id = int(dfc_hits.iloc[0]["id"])
    dfc_descendants = descendants_of_structure_id(dfc_id)
    dfc_whb_overlap = dfc_descendants[["id", "acronym", "name"]].merge(
        roi_by_structure_id,
        left_on="id",
        right_on="structure_id",
        how="inner",
    )
    display(dfc_descendants.loc[dfc_descendants["acronym"].isin(["DFC", "A8", "A9", "A9/46", "A46"]), ["id", "acronym", "name", "structure_id_path"]])
    display(dfc_whb_overlap[["acronym", "name", "region_of_interest_label", "structure_symbol", "structure_name"]])

## Optional: Save Summary Tables

In [ ]:
SAVE_TABLES = False
OUT_DIR = Path("whb_region_exploration_tables")

if SAVE_TABLES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    region_counts.to_csv(OUT_DIR / "region_counts.csv", index=False)
    division_counts.to_csv(OUT_DIR / "division_counts.csv", index=False)
    roi_crosswalk.to_csv(OUT_DIR / "roi_crosswalk.csv", index=False)
    region_matrix_pivot.to_csv(OUT_DIR / "region_counts_by_feature_matrix.csv", index=False)
    print(f"Wrote tables to {OUT_DIR.resolve()}")